In [4]:
import pandas as pd

In [5]:
articles = pd.read_csv('../data/articles.csv')

In [6]:
transactions = pd.read_csv('../data/transactions_train.csv')

In [7]:
articles.info()

<class 'pandas.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   article_id                    105542 non-null  int64
 1   product_code                  105542 non-null  int64
 2   prod_name                     105542 non-null  str  
 3   product_type_no               105542 non-null  int64
 4   product_type_name             105542 non-null  str  
 5   product_group_name            105542 non-null  str  
 6   graphical_appearance_no       105542 non-null  int64
 7   graphical_appearance_name     105542 non-null  str  
 8   colour_group_code             105542 non-null  int64
 9   colour_group_name             105542 non-null  str  
 10  perceived_colour_value_id     105542 non-null  int64
 11  perceived_colour_value_name   105542 non-null  str  
 12  perceived_colour_master_id    105542 non-null  int64
 13  perceived_colour_master_n

In [9]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [10]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [12]:
articles.isnull().sum()

article_id                        0
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_id         0
perceived_colour_value_name       0
perceived_colour_master_id        0
perceived_colour_master_name      0
department_no                     0
department_name                   0
index_code                        0
index_name                        0
index_group_no                    0
index_group_name                  0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
detail_desc                     416
dtype: int64

In [13]:
transactions.isnull().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [14]:
print(articles.columns.tolist())
print(articles['product_type_name'].value_counts().head(20))

['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']
product_type_name
Trousers            11169
Dress               10362
Sweater              9302
T-shirt              7904
Top                  4155
Blouse               3979
Jacket               3940
Shorts               3939
Shirt                3405
Vest top             2991
Underwear bottom     2748
Skirt                2696
Hoodie               2356
Bra                  2212
Socks                1889
Leggings/Tights      1878
Sneakers             1621
Cardigan             1550

Here I am going to filter down to 3 categories: T-shirts, Trousers, and Hoodies. We are going to work with these 3 categories and not let the full H&M catalog bloat the pipeline.

In [15]:
categories = ['T-shirt', 'Trousers', 'Hoodie']

filtered_articles = articles[articles['product_type_name'].isin(categories)]
print(filtered_articles['product_type_name'].value_counts())

product_type_name
Trousers    11169
T-shirt      7904
Hoodie       2356
Name: count, dtype: int64


Here I will write a script that assigns each SKU 6–10 weekly price points

I will use numpy with a seeded elasticity coefficient per category so units sold respond realistically to price changes:
    
    T-shirt → fairly elastic, fast fashion (-1.6)
    Trousers → moderate, more considered purchase (-1.1)
    Hoodie → mid-range elastic (-1.3)

I will export the output to: price_history.csv

In [16]:
import numpy as np
import pandas as pd

# Seeded for reproducibility
rng = np.random.default_rng(42)

# Elasticity coefficients per category
elasticity_map = {
    'T-shirt': -1.6,   # fairly elastic, fast fashion
    'Trousers': -1.1,   # moderate, more considered purchase
    'Hoodie': -1.3     # mid-range elastic
}

# Base prices per category
base_price_map = {
    'T-shirt': 19.99,
    'Trousers': 39.99,
    'Hoodie': 49.99
}

records = []

for category in elasticity_map:
    skus = filtered_articles[filtered_articles['product_type_name'] == category]['article_id'].unique()
    
    for sku in skus:
        base_price = base_price_map[category]
        elasticity = elasticity_map[category]
        base_units = rng.integers(30, 100)
        num_weeks = rng.integers(6, 11)  # 6–10 weekly price points
        
        for week in range(num_weeks):
            # Apply a small random price change each week (-20% to +5%)
            price_change_pct = rng.uniform(-0.20, 0.05)
            price = round(base_price * (1 + price_change_pct), 2)
            
            # Units sold respond to price change via elasticity
            units = int(base_units * ((price / base_price) ** elasticity))
            units = max(units, 1)  # floor at 1
            
            records.append({
                'sku_id': sku,
                'category': category,
                'week': week + 1,
                'price': price,
                'units_sold': units
            })

price_history = pd.DataFrame(records)
print(price_history.head(20))
print(price_history.shape)

       sku_id category  week  price  units_sold
0   189955076  T-shirt     1  18.19          41
1   189955076  T-shirt     2  20.28          35
2   189955076  T-shirt     3  19.48          37
3   189955076  T-shirt     4  16.46          49
4   189955076  T-shirt     5  20.87          33
5   189955076  T-shirt     6  19.80          36
6   189955076  T-shirt     7  19.92          36
7   189955076  T-shirt     8  16.63          48
8   189955076  T-shirt     9  18.24          41
9   194902033  T-shirt     1  20.62          61
10  194902033  T-shirt     2  19.21          69
11  194902033  T-shirt     3  20.10          64
12  194902033  T-shirt     4  18.21          75
13  194902033  T-shirt     5  17.13          83
14  194902033  T-shirt     6  18.76          71
15  194902033  T-shirt     7  16.31          90
16  203027045  T-shirt     1  19.15          96
17  203027045  T-shirt     2  19.78          91
18  203027045  T-shirt     3  17.76         108
19  203027045  T-shirt     4  20.84     

In [17]:
price_history.to_csv('../data/price_history.csv', index=False)

In [18]:
print(price_history['category'].value_counts())
print(price_history['sku_id'].nunique())

category
Trousers    89449
T-shirt     63484
Hoodie      18729
Name: count, dtype: int64
21429


Leaning out # of SKUs for clean, more manageable dataset

In [19]:
# Sample SKUs per category
sku_sample = (
    price_history.groupby('category')['sku_id']
    .apply(lambda x: pd.Series(x.unique()).sample(200, random_state=42))
    .reset_index(drop=True)
)

price_history_sampled = price_history[price_history['sku_id'].isin(sku_sample)]
print(price_history_sampled['category'].value_counts())
print(price_history_sampled['sku_id'].nunique())

category
Hoodie      1604
Trousers    1570
T-shirt     1567
Name: count, dtype: int64
600


In [21]:
price_history_sampled.to_csv('../data/price_history.csv', index=False)

clean_transactions = transactions[transactions['article_id'].isin(price_history_sampled['sku_id'])]
clean_transactions.to_csv('../data/clean_transactions.csv', index=False)

print(clean_transactions.shape)

(147197, 5)
